# Bronze Layer

In [ ]:
# Libraries and variables

import json
import pandas as pd
from pathlib import Path

datapath = Path('../data') #Path to directory containing data
bronze_playlist_path = Path ('../bronze/playlist')
bronze_playlist_path.mkdir(exist_ok = True)

bronze_sliceinfo_path = Path ('../bronze/sliceinfo')
bronze_sliceinfo_path.mkdir(exist_ok = True)

In [ ]:
# writing 'playlist' into ../bronze/
for file in datapath.iterdir():
    with open (file, 'r') as f:
        jread = json.load(f)
    df = pd.json_normalize (jread['playlists'])

    out_name = file.stem.replace('mpd.slice.', 'slices_') + '.parquet'
    df.to_parquet(bronze_playlist_path / out_name)

In [ ]:
# writing 'sliceinfo' into ../bronze/
for file in datapath.iterdir():
    with open (file,'r') as f:
        jread=json.load(f)
    df = pd.json_normalize([jread['info']]) #trying to normalize to the info level

    out_name = file.stem.replace('mpd.slice.', 'slices_') + '.parquet'
    df.to_parquet(bronze_sliceinfo_path / out_name)

In [ ]:
# configure pyspark session

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('plumber') \ 
    .master ('local[*]') \
    .config ("spark.driver.memory", "16g") \
    .getOrCreate()